## Loading data

In [1]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Add parent directory to sys.path so local module 'synthetic' (one level up) can be imported
import sys
from pathlib import Path # for path manipulations
parent_dir = Path.cwd().parent.parent.resolve() # move two levels up from current working directory
if str(parent_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(parent_dir)) # insert at the start of sys.path to prioritize local modules

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv(f'{parent_dir}/XRF_databases/soil/plsda/soil.csv', sep=';') # local copy of Toledo 2022 dataset (os ... indica para omitir o caminho longo)
data = data_complete.loc[:, '1':'15']

In [2]:
# Split dataset by class and create calibration/prediction sets using Kennard-Stone (as in original pipeline)
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '1':'15'], test_size=0.30)  # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '1':'15'], test_size=0.30)  # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True)  # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0])  # target for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0])  # target for prediction set

# preprocessings
import preprocessings as prepr  # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:09:35,102 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:09:35,122 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:09:35,146 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur

In [3]:
# PLS-DA with optimized latent variables
from modeling import pls_optimized

plsda_results = pls_optimized(
    Xcalclass_prep, 
    ycalclass,
    LVmax=4,
    Xpred=Xpredclass_prep,
    ypred=ypredclass,
    aim='classification',
    cv=10
)

# Convenience references used later
pls_model = plsda_results[3]               # fitted PLS model
vip_scores_mat = plsda_results[4]          # VIP scores matrix (features × LV)
y_pred_cont = plsda_results[5].iloc[:, -1] # continuous predictions for Xcalclass (used for MI/Cov)

## Spectral cuts (domain knowledge)

In [4]:
# establishing spectral cuts based on expert knowledge of XRF spectra
spectral_cuts = [
('background1', 1.0, 1.33),
('Al', 1.34, 1.63),
('Si', 1.64, 1.86),
('P', 1.87, 2.10),
('background2', 2.11, 2.19),
('S', 2.20, 2.44),
('background3', 2.45, 2.55),
('Rh L + Ar', 2.56, 3.10),
('background4', 3.11, 3.21),
('K', 3.22, 3.42),
('background5', 3.43, 3.53),
('Ca ka', 3.54, 3.84),
('Ca kb', 3.92, 4.14),
('background6', 4.15, 4.37),
('Ti ka', 4.38, 4.66),
('background7', 4.67, 4.75),
('Ti kb', 4.76, 5.12),
('Cr', 5.13, 5.77),
('Mn', 5.78, 6.02),
('background8', 6.03, 6.13),
('Fe ka', 6.14, 6.68),
('background9', 6.69, 6.80),
('Fe kb', 6.81, 7.30),
('background10', 7.31, 7.91),
('Cu', 7.92, 8.20),
('background11', 8.21, 10.69),
('Fe ka + Ti ka', 10.7, 11.14),
('background12', 11.15, 12.55),
('sum Fe' , 12.56, 13.1),
('background13', 13.11, 15.0)
]

import explaining as exp
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_sums_df = exp.aggregate_spectral_zones(spectral_zones_class, aggregator='extreme')
predicates_quantiles = exp.predicates_by_quantiles(zone_sums_df, [0.2, 0.4, 0.6, 0.8])
co_occurrence_matrix_df = predicates_quantiles[2]
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_sums_df,
    y_predicted_numeric=y_pred_cont
)

## VIP, Regression Coefficients e SHAP (como no original)

In [5]:
# VIP scores por energia
vip_scores_df = pd.DataFrame({
    'energy': vip_scores_mat.T.index,
    'VIP_Score': vip_scores_mat.T.iloc[:,0].values
})
vip_scores_df = vip_scores_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in vip_scores_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
vip_scores_df['Zone'] = vip_scores_df['energy'].map(energy_to_zone_vip)
vip_scores_unique_df = vip_scores_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
vip_scores_unique_df = vip_scores_unique_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# Coeficientes de regressão do PLS
reg_vet = pd.DataFrame(pls_model.coef_, columns=pls_model.feature_names_in_).T
reg_vet.insert(0, 'energy', reg_vet.index)
reg_vet = reg_vet.reset_index(drop=True)
reg_vet.columns = ['energy','Reg_coef']
reg_vet['Abs_Reg_coef'] = reg_vet['Reg_coef'].abs()
reg_vet = reg_vet.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)
energy_to_zone_reg = {}
for zone_name, start, end in spectral_cuts:
    for e in reg_vet['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_reg[e] = zone_name
reg_vet['Zone'] = reg_vet['energy'].map(energy_to_zone_reg)
reg_vet_unique_df = reg_vet.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
reg_vet_unique_df = reg_vet_unique_df.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)

# # vamos agora extrair as variaveis mais importantes atraves do método SHAP
# import shap

# # Para PLSRegression, usamos KernelExplainer porque não há explainer dedicado muito rápido
# explainer_pls = shap.KernelExplainer(plsda_results[3].predict, Xcalclass_prep)
# shap_values_pls = explainer_pls(Xcalclass_prep)

# shap_global_importance = pd.DataFrame({
#     'energy': Xpredclass_prep.columns,
#     'Mean_Abs_SHAP': np.abs(shap_values_pls.values).mean(axis=0)}) # tomando a importancia global como a media dos valores absolutos dos valores SHAP para cada feature
# shap_global_importance.sort_values(by='Mean_Abs_SHAP', ascending=False, inplace=True)

# # vamos gerar uma nova coluna em shap_global_importance com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
# energy_to_zone_shap = {}
# for zone_name, start, end in spectral_cuts:
#     for i in shap_global_importance['energy']:
#         i_float = float(i)
#         if start <= i_float <= end:
#             energy_to_zone_shap[i] = zone_name
# shap_global_importance['Zone'] = shap_global_importance['energy'].map(energy_to_zone_shap)

# # agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
# shap_unique_df = shap_global_importance.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
# shap_unique_df = shap_unique_df.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
# shap_unique_df.to_csv('shap_soil.csv', index=False, sep=';')
shap_unique_df = pd.read_csv('shap_soil.csv', sep=';') # loading previously saved shap_unique_df

# **Comparando com o bagging**

In [6]:
# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 42]

all_results = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_predicted_numeric = plsda_results[5].iloc[:, -1] # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=20,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    # Calcular MI
    mi_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance',
        threshold=0.001, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    # Salvar no dicionário principal
    all_results[seed] = {
        'bags_result': bags_result_seed,
        'mi_results_dict': mi_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")

# Construir grafo para esta semente
    DG = exp.build_predicate_graph(
        bags_result=all_results[seed]['bags_result'],
        mi_results_dict=all_results[seed]['mi_results_dict'],
        co_occurrence_matrix_df=co_occurrence_matrix_df,
        predicates_df=predicates_quantiles[0],
        random_state=seed,
        show_details=True
    )

#     DG = exp.build_fold_predicate_graph(
#     bags_result=all_results[seed]['bags_result'],
#     mi_results_dict=all_results[seed]['mi_results_dict'],
#     predicates_df=predicates_quantiles[0],
#     random_state=42,
#     show_details=True,
#     normalize_weights=True,
#     weight_mode='ranking',
#     co_occurrence_matrix=co_occurrence_matrix_df,
#     apply_confidence_multiplier=True,
#     accumulate_cooccurrence_weights=True
# )

    # Armazenar grafo
    graphs_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_by_seed[seed] = lrc_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_df_seed = lrc_by_seed[seed].rename(columns={'Node': f'Predicate_Seed_{seed}'})
    lrc_all_seeds_df = pd.concat([lrc_all_seeds_df, lrc_df_seed[[f'Predicate_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_unique_by_seed = {}
for seed, lrc_df in lrc_by_seed.items():
    lrc_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_unique_df = lrc_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_unique_by_seed[seed] = lrc_unique_df

lrc_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 183 | Descartados: 57
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 183 | Descartados: 57
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 182 | Descartados: 58
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 184 | Descartados: 56
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 183 | Descartados: 57
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 182 | Descartados: 58
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 180 | Descartados: 60
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 182 | Descartados: 58
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 183 | Descartados: 57
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 181 | Descartados: 59
Bag_11 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 183 | Descartados: 57
Bag_12 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 183 | Descartados: 57


C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning: divide by zero encountered in scalar divide
  return total_weight / d.get(weight, 1)



Processando LRC do grafo...


C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning: divide by zero encountered in scalar divide
  return total_weight / d.get(weight, 1)



Processando LRC do grafo...


C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning: divide by zero encountered in scalar divide
  return total_weight / d.get(weight, 1)


,Predicate_Seed_0,Predicate_Seed_1,Predicate_Seed_42
0,Ca ka > -0.72,Ca ka > -0.47,Ca ka > -0.22
1,Ca ka > -0.22,Ca ka > -0.72,Ca ka > -0.72
2,Ca ka > -0.47,Fe ka > -1.50,Ca ka > -0.47
3,Fe ka > -1.50,Fe ka > -0.44,Fe ka > -1.50
4,Fe ka <= 1.45,Ca ka > -0.22,Fe ka <= 1.45
...,...,...,...
194,NaN,NaN,K <= -0.21
195,NaN,NaN,K > 0.21
196,NaN,NaN,P <= -0.16
197,NaN,NaN,Class_A


# Kennard-Stone + Round-Robin k-fold

## Duas Estratégias Disponíveis

### 1. Estratégia GLOBAL (`per_predicate=False`) - Original
- KS é aplicado **globalmente** em todas as amostras do dataset
- Distribui amostras via round-robin para k folds
- **Todos os predicados compartilham os mesmos folds**
- Predicados com cobertura < min_samples são eliminados

**Vantagens:**
- Consistência: mesmas amostras nos mesmos folds para todos os predicados
- Comparabilidade direta entre predicados
- Menor custo computacional (KS executado uma única vez)

**Desvantagens:**
- Predicados com baixa cobertura global podem ser eliminados
- A diversidade do KS é otimizada globalmente, não por predicado

---

### 2. Estratégia PER-PREDICATE (`per_predicate=True`) - Nova
- KS é aplicado **individualmente** para cada predicado
- Considera apenas as amostras que satisfazem cada predicado
- **Cada predicado tem seus próprios folds independentes**
- Resultados são combinados ao final

**Vantagens:**
- Maximiza representatividade dentro de cada predicado
- Mais amostras válidas por predicado (menos eliminações)
- Independência estatística entre predicados
- Diversidade otimizada para cada predicado individualmente

**Desvantagens:**
- Folds inconsistentes entre predicados (amostra X pode estar no Fold_1 para um predicado e Fold_3 para outro)
- Maior custo computacional (KS executado N vezes)
- Combinação de resultados requer cuidado na interpretação

**Solução técnica para KS unidimensional:**
- KS precisa de múltiplas variáveis para calcular distâncias
- Solução: adicionar o índice normalizado da amostra como segunda coluna
- Isso é determinístico e representa a "posição temporal" da amostra

In [7]:
import ks_folding as ksf

folds_result = ksf.kfold_predicates_roundrobin(
    zone_sums_df=zone_sums_df,
    y_predicted_numeric=y_pred_cont,
    predicates_df=predicates_quantiles[0],
    k_folds=2,
    min_samples_ratio=0.001,  # 60% das amostras do fold
    verbose=True,
    per_predicate=False # escolhe entre fazer o fold por predicado ou globalmente (que faz o fold para todos os predicados juntos)
)

# Adiciona classe prevista (A/B) em cada DataFrame de predicado
for fold_name, pred_dict in folds_result.items():
    for rule, df_info in pred_dict.items():
        df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B')

mi_results_dict = exp.calculate_predicate_metrics(
    bags_result=folds_result,
    metric='covariance',
    threshold=0.001,
    #n_neighbors=5
)

max_len = max(len(mi_df['Predicate']) for mi_df in mi_results_dict.values())
padded_dict = {
    f'Predicate_{fold}': list(mi_df['Predicate']) + [None]*(max_len - len(mi_df['Predicate']))
    for fold, mi_df in mi_results_dict.items()
}
all_cov_results = pd.DataFrame(padded_dict)
all_cov_results

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:04,307 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:04,310 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.



Configuração KS + Round-Robin
Estratégia: GLOBAL (compartilhada)
Total de amostras: 148
Número de folds: 2
Amostras por fold (aprox.): 74
Mínimo de amostras por predicado: 2 (0% do fold)



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(



=== Resumo (Estratégia GLOBAL - SEM estratificação) ===
Predicados totais: 240
Predicados eliminados (em pelo menos 1 fold): 0
Folds criados: 2
  Fold_1: 240 predicados válidos
  Fold_2: 240 predicados válidos
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001


,Predicate_Fold_1,Predicate_Fold_2
0,Ca ka > -0.72,Ca ka > 0.58
1,Fe ka <= 1.45,Ca ka > -0.22
2,Fe ka > -1.50,Ca ka > -0.47
3,Ca ka > -0.47,Ca ka > -0.72
4,Fe ka <= 0.86,Ca kb > 0.19
...,...,...
204,background3 <= 0.05,None
205,background4 <= -0.05,None
206,background4 <= -0.09,None
207,background3 <= -0.06,None


In [8]:
DG = exp.build_fold_predicate_graph(
    bags_result=folds_result,           # Resultado dos folds (KS + Round-Robin)
    mi_results_dict=mi_results_dict,    # Rankings de Covariância por fold
    predicates_df=predicates_quantiles[0],  # DataFrame com metadados dos predicados
    random_state=42,                    # Semente para reprodutibilidade
    show_details=True,                  # Mostra detalhes da resolução
    normalize_weights=True,            # False = peso inteiro | True = peso [1/k, 1]
    weight_mode='cooccurrence',         # 'ranking' ou 'cooccurrence'
    co_occurrence_matrix=co_occurrence_matrix_df,  # Necessário se weight_mode='cooccurrence'
    apply_confidence_multiplier=True,    # True = peso × score | False = só co-ocorrência
    accumulate_cooccurrence_weights=True  # Nova opção!
)
DG

# Calcula LRC para cada nó e compõe DataFrame
lrc_ks_df = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
# Seleciona apenas uma ocorrência por zona (maior LRC)
lrc_ks_unique_df = lrc_ks_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_ks_df

 AVISO: normalize_weights=True ignorado em weight_mode='cooccurrence'
   (Pesos de co-ocorrência representam contagens reais de amostras)

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: COOCCURRENCE
Fonte: Matriz de co-ocorrência global
Sub-estratégia: ACUMULATIVA (soma valores da matriz)

Folds processados: 2
Arestas criadas (antes de resolver bidirecionais): 409

RESOLUÇÃO DE ARESTAS BIDIRECIONAIS
Total de pares bidirecionais encontrados: 0
Critério de desempate: PESO ACUMULADO (soma das co-ocorrências locais)

APLICAÇÃO DO MULTIPLICADOR DE CONFIANÇA
Fórmula: peso_final = co_ocorrência × score_de_confiança
  Ca ka > -0.72 → Fe ka <= 1.45: 98 × 209 = 20482.00
  Ca ka > -0.72 → Ca kb > 0.19: 30 × 200 = 6000.00
  Fe ka <= 1.45 → Fe ka > -1.50: 89 × 208 = 18512.00
  Fe ka <= 1.45 → Ti ka > 0.91: 18 × 188 = 3384.00
  Fe ka > -1.50 → Ca ka > -0.47: 65 × 207 = 13455.00
  Fe ka > -1.50 → Ti ka > -0.80: 97 × 197 = 19109.00
  Ca ka > -0.47 → Fe ka <= 0.86: 61 × 206 = 12566.00
  Ca ka > -0.47 

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning: divide by zero encountered in scalar divide
  return total_weight / d.get(weight, 1)


,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Mn > -0.35,3.366031,Mn,-0.35,>
1,Fe kb <= 0.57,2.707959,Fe kb,0.57,<=
2,Ti ka <= 0.91,2.344915,Ti ka,0.91,<=
3,sum Fe <= 0.15,2.324404,sum Fe,0.15,<=
4,background13 <= 0.18,2.191017,background13,0.18,<=
...,...,...,...,...,...
228,Cu <= 0.15,0.743260,Cu,0.15,<=
229,Rh L + Ar > 0.19,0.696638,Rh L + Ar,0.19,>
230,Fe ka + Ti ka <= -0.13,0.000016,Fe ka + Ti ka,-0.13,<=
231,Class_A,0.000000,None,None,None


# **Variando o numero de folds - modo cooccurrence**

In [9]:
# Loop para gerar múltiplos grafos e DataFrames LRC para diferentes valores de k_folds

folds_list = [2, 3, 4, 5, 6]  # Exemplo de diferentes valores de k_folds

graphs_ks_by_fold = {}
lrc_ks_df_by_fold = {}
lrc_ks_unique_df_by_fold = {}
all_fold_results_cooc = {}

for k_folds in folds_list:
    print(f"\n{'='*70}")
    print(f"Processando k_folds: {k_folds}")
    print(f"{'='*70}\n")

    # Geração dos folds
    folds_result = ksf.kfold_predicates_roundrobin(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_pred_cont,
        predicates_df=predicates_quantiles[0],
        k_folds=k_folds,
        min_samples_ratio=0.001,
        verbose=True,
        per_predicate=True
    )

    # Adiciona classe prevista (A/B) em cada DataFrame de predicado
    for fold_name, pred_dict in folds_result.items():
        for rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B')

    # Calcula métricas de covariância
    mi_results_dict = exp.calculate_predicate_metrics(
        bags_result=folds_result,
        metric='covariance',
        threshold=0.001,
    )

        # Salvar no dicionário principal
    all_fold_results_cooc[k_folds] = {
        'cov_results_dict': mi_results_dict
    }

    # Constrói o grafo
    DG = exp.build_fold_predicate_graph(
        bags_result=folds_result,
        mi_results_dict=mi_results_dict,
        predicates_df=predicates_quantiles[0],
        random_state=42,
        show_details=True,
        normalize_weights=True,
        weight_mode='cooccurrence',
        co_occurrence_matrix=co_occurrence_matrix_df,
        apply_confidence_multiplier=True,
        accumulate_cooccurrence_weights=False
    )

    # Calcula LRC para cada nó e compõe DataFrame
    lrc_ks_df = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_ks_unique_df = lrc_ks_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)

    # Salva nos dicionários
    graphs_ks_by_fold[k_folds] = DG
    lrc_ks_df_by_fold[k_folds] = lrc_ks_df
    lrc_ks_unique_df_by_fold[k_folds] = lrc_ks_unique_df

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:06,080 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:06,084 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur


Processando k_folds: 2

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 148
Número de folds: 2
Amostras por fold (aprox.): 74
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:06,277 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:06,284 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 240
Predicados válidos: 240
Predicados eliminados: 0
Folds criados: 2

Estatísticas por predicado válido:
  'background1 <= -0.11': 26 amostras, folds: [13, 13]
  'background1 > -0.11': 122 amostras, folds: [61, 61]
  'background1 <= -0.07': 63 amostras, folds: [32, 31]
  'background1 > -0.07': 85 amostras, folds: [43, 42]
  'background1 <= 0.09': 94 amostras, folds: [47, 47]
  ... e mais 235 predicados

Predicados por fold:
  Fold_1: 240 predicados
  Fold_2: 240 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: normalize_weights=True ignorado em weight_mode='cooccurrence'
   (Pesos de co-ocorrência representam contagens reais de amostras)

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: COOCCURRENCE
Fonte: Matriz de co-ocorrência global
Sub-estratégia: NÃO-ACUMULATIVA (peso fixo da matriz)

Folds processados: 2
Arestas criadas (antes de resolver bidirecionais): 408

RESOLUÇÃO DE ARESTAS B

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning: divide by zero encountered in scalar divide
  return total_weight / d.get(weight, 1)
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:10,568 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:10,570 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarni

...

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning: divide by zero encountered in scalar divide
  return total_weight / d.get(weight, 1)
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:25,354 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:25,356 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarni


Processando k_folds: 6

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 148
Número de folds: 6
Amostras por fold (aprox.): 24
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:25,562 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:25,565 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 240
Predicados válidos: 240
Predicados eliminados: 0
Folds criados: 6

Estatísticas por predicado válido:
  'background1 <= -0.11': 26 amostras, folds: [5, 5, 4, 4, 4, 4]
  'background1 > -0.11': 122 amostras, folds: [21, 21, 20, 20, 20, 20]
  'background1 <= -0.07': 63 amostras, folds: [11, 11, 11, 10, 10, 10]
  'background1 > -0.07': 85 amostras, folds: [15, 14, 14, 14, 14, 14]
  'background1 <= 0.09': 94 amostras, folds: [16, 16, 16, 16, 15, 15]
  ... e mais 235 predicados

Predicados por fold:
  Fold_1: 240 predicados
  Fold_2: 240 predicados
  Fold_3: 240 predicados
  Fold_4: 240 predicados
  Fold_5: 240 predicados
  Fold_6: 240 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: normalize_weights=True ignorado em weight_mode='cooccurrence'
   (Pesos de co-ocorrência representam contagens reais de amostras)

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: COOCCURRENCE
Fonte: Matriz de c

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning: divide by zero encountered in scalar divide
  return total_weight / d.get(weight, 1)


In [10]:
features_importance = pd.DataFrame({
    'Vip' : vip_scores_unique_df['Zone'].iloc[:10].values,
    'Reg_coef' : reg_vet_unique_df['Zone'].iloc[:10].values,
    'Shap' : shap_unique_df['Zone'].iloc[:10].values
    })

for k_folds, lrc_unique_df in lrc_ks_unique_df_by_fold.items():
     features_importance[f'LRC_kfold_{k_folds}'] = lrc_unique_df['Zone'].iloc[:10].values

for seed, lrc_unique_df in lrc_unique_by_seed.items():
    features_importance[f'LRC_Seed_{seed}'] = lrc_unique_df['Zone'].iloc[:10].values

# vamos exportar o df features_importance para um arquivo excel onde vamos nomear a sheet de acordo com as comparacoes feitas
features_importance.to_excel('features_importance_soil.xlsx', index=False, sheet_name='KS_fold_cooc')
features_importance

,Vip,Reg_coef,Shap,LRC_kfold_2,LRC_kfold_3,LRC_kfold_4,LRC_kfold_5,LRC_kfold_6,LRC_Seed_0,LRC_Seed_1,LRC_Seed_42
0,Ca ka,Si,Ca ka,Ca kb,Ca ka,Ca ka,Ca ka,Ca ka,Ca ka,Ca ka,Ca ka
1,Fe ka,Mn,Mn,Mn,Ca kb,Ti ka,Mn,Ti ka,Fe ka,Fe ka,Fe ka
2,Mn,Ca ka,Si,Ti ka,Fe ka,Fe ka,Si,background11,Ca kb,Fe kb,Al
3,Si,Ti ka,Fe ka,Ca ka,Fe kb,Mn,K,Rh L + Ar,Fe kb,Ca kb,Ca kb
4,Fe kb,Fe ka,Ti ka,Fe kb,Cr,Ca kb,Ti ka,P,Al,Mn,Ti kb
5,Ti ka,P,Ca kb,Si,sum Fe,Ti kb,Fe ka,Ca kb,Cr,Si,background1
6,Ca kb,Al,Fe kb,sum Fe,S,Al,Fe kb,Fe ka,background13,background13,Ti ka
7,Al,Ca kb,K,Fe ka,background7,S,Ca kb,Si,Si,K,Mn
8,K,background11,P,background6,Rh L + Ar,Si,Ti kb,Fe kb,Mn,Ti ka,sum Fe
9,sum Fe,K,background10,Cr,Ti ka,P,background3,Mn,Ti ka,Al,Si


In [11]:
# RBO (Rank-Biased Overlap) para comparar rankings
import rbo
rbo_results = {}
reference_list = features_importance['Vip'].tolist()
methods = ['Reg_coef', 'Shap'] + [f'LRC_kfold_{fold}' for fold in folds_list] + [f'LRC_Seed_{seed}' for seed in random_seeds]
for method in methods:
    compare_list = features_importance[method].tolist()
    score = rbo.RankingSimilarity(reference_list, compare_list).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method','RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_results.to_excel('rbo_soil.xlsx', index=False, sheet_name='KS_fold_cooc')
rbo_results

,Reference,Method,RBO_Score
8,Vip,LRC_Seed_1,0.834142
7,Vip,LRC_Seed_0,0.789713
1,Vip,Shap,0.787512
9,Vip,LRC_Seed_42,0.764982
5,Vip,LRC_kfold_5,0.742339
4,Vip,LRC_kfold_4,0.722673
3,Vip,LRC_kfold_3,0.670350
6,Vip,LRC_kfold_6,0.566552
0,Vip,Reg_coef,0.344781
2,Vip,LRC_kfold_2,0.260692


# **Variando o numero de folds - modo ranking**

In [12]:
# Loop para gerar múltiplos grafos e DataFrames LRC para diferentes valores de k_folds

folds_list = [2, 3, 4, 5, 6]  # Exemplo de diferentes valores de k_folds

graphs_ks_by_fold = {}
lrc_ks_df_by_fold = {}
lrc_ks_unique_df_by_fold = {}
all_fold_results_rank = {}

for k_folds in folds_list:
    print(f"\n{'='*70}")
    print(f"Processando k_folds: {k_folds}")
    print(f"{'='*70}\n")

    # Geração dos folds
    folds_result = ksf.kfold_predicates_roundrobin(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_pred_cont,
        predicates_df=predicates_quantiles[0],
        k_folds=k_folds,
        min_samples_ratio=0.001,
        verbose=True,
        per_predicate=True
    )

    # Adiciona classe prevista (A/B) em cada DataFrame de predicado
    for fold_name, pred_dict in folds_result.items():
        for rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B')

    # Calcula métricas de covariância
    mi_results_dict = exp.calculate_predicate_metrics(
        bags_result=folds_result,
        metric='covariance',
        threshold=0.001,
    )

        # Salvar no dicionário principal
    all_fold_results_rank[k_folds] = {
        'cov_results_dict': mi_results_dict
    }

    # Constrói o grafo
    DG = exp.build_fold_predicate_graph(
        bags_result=folds_result,
        mi_results_dict=mi_results_dict,
        predicates_df=predicates_quantiles[0],
        random_state=42,
        show_details=True,
        normalize_weights=True,
        weight_mode='ranking',
        co_occurrence_matrix=co_occurrence_matrix_df,
        apply_confidence_multiplier=True,
        accumulate_cooccurrence_weights=True
    )

    # Calcula LRC para cada nó e compõe DataFrame
    lrc_ks_df = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_ks_unique_df = lrc_ks_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)

    # Salva nos dicionários
    graphs_ks_by_fold[k_folds] = DG
    lrc_ks_df_by_fold[k_folds] = lrc_ks_df
    lrc_ks_unique_df_by_fold[k_folds] = lrc_ks_unique_df

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:31,598 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:31,602 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur


Processando k_folds: 2

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 148
Número de folds: 2
Amostras por fold (aprox.): 74
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:31,802 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:31,804 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 240
Predicados válidos: 240
Predicados eliminados: 0
Folds criados: 2

Estatísticas por predicado válido:
  'background1 <= -0.11': 26 amostras, folds: [13, 13]
  'background1 > -0.11': 122 amostras, folds: [61, 61]
  'background1 <= -0.07': 63 amostras, folds: [32, 31]
  'background1 > -0.07': 85 amostras, folds: [43, 42]
  'background1 <= 0.09': 94 amostras, folds: [47, 47]
  ... e mais 235 predicados

Predicados por fold:
  Fold_1: 240 predicados
  Fold_2: 240 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: apply_confidence_multiplier=True ignorado em weight_mode='ranking'
   (Multiplicador de confiança só se aplica ao modo 'cooccurrence')
 AVISO: accumulate_cooccurrence_weights=True ignorado em weight_mode='ranking'
   (Acumulação de co-ocorrências só se aplica ao modo 'cooccurrence')

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: RANKING

Folds processados: 2
Arestas criadas (ante

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:36,036 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:36,038 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur

...


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 240
Predicados válidos: 240
Predicados eliminados: 0
Folds criados: 5

Estatísticas por predicado válido:
  'background1 <= -0.11': 26 amostras, folds: [6, 5, 5, 5, 5]
  'background1 > -0.11': 122 amostras, folds: [25, 25, 24, 24, 24]
  'background1 <= -0.07': 63 amostras, folds: [13, 13, 13, 12, 12]
  'background1 > -0.07': 85 amostras, folds: [17, 17, 17, 17, 17]
  'background1 <= 0.09': 94 amostras, folds: [19, 19, 19, 19, 18]
  ... e mais 235 predicados

Predicados por fold:
  Fold_1: 240 predicados
  Fold_2: 240 predicados
  Fold_3: 240 predicados
  Fold_4: 240 predicados
  Fold_5: 240 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: apply_confidence_multiplier=True ignorado em weight_mode='ranking'
   (Multiplicador de confiança só se aplica ao modo 'cooccurrence')
 AVISO: accumulate_cooccurrence_weights=True ignorado em weight_mode='ranking'
   (Acumulação de co-ocorrências só se a

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:50,251 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:50,253 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur


Processando k_folds: 6

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 148
Número de folds: 6
Amostras por fold (aprox.): 24
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:50,457 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:10:50,467 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 240
Predicados válidos: 240
Predicados eliminados: 0
Folds criados: 6

Estatísticas por predicado válido:
  'background1 <= -0.11': 26 amostras, folds: [5, 5, 4, 4, 4, 4]
  'background1 > -0.11': 122 amostras, folds: [21, 21, 20, 20, 20, 20]
  'background1 <= -0.07': 63 amostras, folds: [11, 11, 11, 10, 10, 10]
  'background1 > -0.07': 85 amostras, folds: [15, 14, 14, 14, 14, 14]
  'background1 <= 0.09': 94 amostras, folds: [16, 16, 16, 16, 15, 15]
  ... e mais 235 predicados

Predicados por fold:
  Fold_1: 240 predicados
  Fold_2: 240 predicados
  Fold_3: 240 predicados
  Fold_4: 240 predicados
  Fold_5: 240 predicados
  Fold_6: 240 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: apply_confidence_multiplier=True ignorado em weight_mode='ranking'
   (Multiplicador de confiança só se aplica ao modo 'cooccurrence')
 AVISO: accumulate_cooccurrence_weights=True ignorado em weight_mode='ranki

In [13]:
features_importance = pd.DataFrame({
    'Vip' : vip_scores_unique_df['Zone'].iloc[:10].values,
    'Reg_coef' : reg_vet_unique_df['Zone'].iloc[:10].values,
    'Shap' : shap_unique_df['Zone'].iloc[:10].values
    })

for k_folds, lrc_unique_df in lrc_ks_unique_df_by_fold.items():
    features_importance[f'LRC_kfold_{k_folds}'] = lrc_unique_df['Zone'].iloc[:10].values

for seed, lrc_unique_df in lrc_unique_by_seed.items():
    features_importance[f'LRC_Seed_{seed}'] = lrc_unique_df['Zone'].iloc[:10].values
    
with pd.ExcelWriter('features_importance_soil.xlsx', mode='a', engine='openpyxl') as writer:
    features_importance.to_excel(writer, sheet_name='KS_fold_rank', index=False)
features_importance

,Vip,Reg_coef,Shap,LRC_kfold_2,LRC_kfold_3,LRC_kfold_4,LRC_kfold_5,LRC_kfold_6,LRC_Seed_0,LRC_Seed_1,LRC_Seed_42
0,Ca ka,Si,Ca ka,Ca ka,Ca kb,Fe ka,Ca ka,Fe ka,Ca ka,Ca ka,Ca ka
1,Fe ka,Mn,Mn,Fe ka,Fe ka,Ca ka,Ti ka,Ti ka,Fe ka,Fe ka,Fe ka
2,Mn,Ca ka,Si,Ti ka,Ca ka,Ti ka,Fe kb,Ca ka,Ca kb,Fe kb,Al
3,Si,Ti ka,Fe ka,Ca kb,Ti ka,Rh L + Ar,Si,Fe kb,Fe kb,Ca kb,Ca kb
4,Fe kb,Fe ka,Ti ka,Fe kb,Fe kb,Ca kb,Mn,Ca kb,Al,Mn,Ti kb
5,Ti ka,P,Ca kb,Mn,Mn,Mn,sum Fe,P,Cr,Si,background1
6,Ca kb,Al,Fe kb,Ti kb,K,Si,Ca kb,Mn,background13,background13,Ti ka
7,Al,Ca kb,K,sum Fe,Si,P,Fe ka,background11,Si,K,Mn
8,K,background11,P,Cr,Rh L + Ar,Ti kb,P,Al,Mn,Ti ka,sum Fe
9,sum Fe,K,background10,Al,Cr,K,Ti kb,Ti kb,Ti ka,Al,Si


In [14]:
# RBO (Rank-Biased Overlap) para comparar rankings
import rbo
rbo_results = {}
reference_list = features_importance['Vip'].tolist()
methods = ['Reg_coef', 'Shap'] + [f'LRC_kfold_{fold}' for fold in folds_list] + [f'LRC_Seed_{seed}' for seed in random_seeds]
for method in methods:
    compare_list = features_importance[method].tolist()
    score = rbo.RankingSimilarity(reference_list, compare_list).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method','RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)
with pd.ExcelWriter('rbo_soil.xlsx', mode='a', engine='openpyxl') as writer:
    rbo_results.to_excel(writer, sheet_name='KS_fold_rank', index=False)
rbo_results

,Reference,Method,RBO_Score
8,Vip,LRC_Seed_1,0.834142
2,Vip,LRC_kfold_2,0.814682
7,Vip,LRC_Seed_0,0.789713
1,Vip,Shap,0.787512
9,Vip,LRC_Seed_42,0.764982
5,Vip,LRC_kfold_5,0.680098
4,Vip,LRC_kfold_4,0.490662
3,Vip,LRC_kfold_3,0.416614
6,Vip,LRC_kfold_6,0.401990
0,Vip,Reg_coef,0.344781


# **Comparando com rankings medios**

In [15]:
ranking_predicate_mean_seeds = {}
ranking_predicate_mean_unique_seeds = {}

for seed in random_seeds:
    mi_results_dict_seed = all_results[seed]['mi_results_dict']
    ranking_predicate_mean, ranking_predicate_mean_unique = exp.calculate_predicate_ranking_mean(
        mi_results_dict_seed, 
        return_unique_zones=True
    )
    ranking_predicate_mean_seeds[seed] = ranking_predicate_mean
    ranking_predicate_mean_unique_seeds[seed] = ranking_predicate_mean_unique
# Exibindo resultados para uma semente específica

In [16]:
ranking_predicate_mean_folds = {}
ranking_predicate_mean_unique_folds = {}

for fold in folds_list:
    mi_results_dict_fold = all_fold_results_cooc[fold]['cov_results_dict']
    ranking_predicate_mean, ranking_predicate_mean_unique = exp.calculate_predicate_ranking_mean(
        mi_results_dict_fold, 
        return_unique_zones=True
    )
    ranking_predicate_mean_folds[fold] = ranking_predicate_mean
    ranking_predicate_mean_unique_folds[fold] = ranking_predicate_mean_unique
# Exibindo resultados para uma semente específica

In [17]:
# Build the dictionary step by step to avoid mixing comprehension and static entries
features_dict = {
    'Vip' : vip_scores_unique_df['Zone'].iloc[:10].values,
    'Reg_coef' : reg_vet_unique_df['Zone'].iloc[:10].values,
    'Shap' : shap_unique_df['Zone'].iloc[:10].values
}
# Add the predicate rankings from seeds
for seed, ranking_df in ranking_predicate_mean_unique_seeds.items():
    features_dict[f'Cov_Mean_Seed_{seed}'] = ranking_df['Zone'].iloc[:10].values
# Add the predicate rankings from folds
for fold, ranking_df in ranking_predicate_mean_unique_folds.items():
    features_dict[f'Cov_Mean_Fold_{fold}'] = ranking_df['Zone'].iloc[:10].values
features_importance = pd.DataFrame(features_dict)
features_importance

,Vip,Reg_coef,Shap,Cov_Mean_Seed_0,Cov_Mean_Seed_1,Cov_Mean_Seed_42,Cov_Mean_Fold_2,Cov_Mean_Fold_3,Cov_Mean_Fold_4,Cov_Mean_Fold_5,Cov_Mean_Fold_6
0,Ca ka,Si,Ca ka,Ca ka,Ca ka,Ca ka,Fe ka,Ca ka,Ca ka,Ca ka,Ca ka
1,Fe ka,Mn,Mn,Fe ka,Fe ka,Fe ka,Ca ka,Fe ka,Fe ka,Fe ka,Fe ka
2,Mn,Ca ka,Si,Mn,Mn,Ca kb,Fe kb,Ti ka,Mn,Fe kb,Mn
3,Si,Ti ka,Fe ka,Ca kb,Fe kb,Mn,Ti ka,Mn,Fe kb,Mn,Ti ka
4,Fe kb,Fe ka,Ti ka,Fe kb,Ca kb,Fe kb,Mn,Fe kb,Ti ka,Ti ka,Fe kb
5,Ti ka,P,Ca kb,Ti ka,Ti ka,Ti ka,Ca kb,Ca kb,Si,Ca kb,Si
6,Ca kb,Al,Fe kb,Si,Si,Si,Si,Si,Ca kb,K,Ca kb
7,Al,Ca kb,K,K,K,K,Ti kb,Al,Al,Al,Al
8,K,background11,P,Al,Al,Al,K,K,K,Si,K
9,sum Fe,K,background10,sum Fe,P,sum Fe,Al,P,Ti kb,P,Ti kb


In [18]:
import rbo
rbo_results = {}
reference_list = features_importance['Vip'].tolist()
methods = ['Reg_coef', 'Shap'] + [f'Cov_Mean_Seed_{seed}' for seed in random_seeds] + [f'Cov_Mean_Fold_{fold}' for fold in folds_list]
for method in methods:
    compare_list = features_importance[method].tolist()
    score = rbo.RankingSimilarity(reference_list, compare_list).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method','RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_results

,Reference,Method,RBO_Score
7,Vip,Cov_Mean_Fold_4,0.930411
9,Vip,Cov_Mean_Fold_6,0.930411
2,Vip,Cov_Mean_Seed_0,0.920130
3,Vip,Cov_Mean_Seed_1,0.918919
6,Vip,Cov_Mean_Fold_3,0.873007
4,Vip,Cov_Mean_Seed_42,0.871130
8,Vip,Cov_Mean_Fold_5,0.864877
1,Vip,Shap,0.787512
5,Vip,Cov_Mean_Fold_2,0.542272
0,Vip,Reg_coef,0.344781
